# Level 1 protocol and scale audit

This notebook verifies the paired 8-layer experiment before training.

In [ ]:
from pathlib import Path
import yaml

repo = Path.cwd()
while not (repo / 'level_1_baseline').exists() and repo != repo.parent:
    repo = repo.parent
b = yaml.safe_load((repo / 'level_1_baseline/configs/level1.yaml').read_text())
w = yaml.safe_load((repo / 'level_1_wwpgd/configs/level1.yaml').read_text())
for section in ('model', 'training', 'analysis'):
    assert b[section] == w[section], section
assert w['wwpgd']['apply_mode'] == 'event_projection'
assert w['wwpgd']['interval'] == 1
assert w['wwpgd']['target_alpha'] == 2.0
b

In [ ]:
m = b['model']; t = b['training']
V, T, L, d = m['vocab_size'], m['block_size'], m['n_layer'], m['n_embd']
# Decoder-only GPT estimate with tied embeddings: token + position embeddings,
# 12 d^2 parameters per block, and final LayerNorm.
params = V*d + T*d + L*(12*d*d + 4*d) + 2*d
tokens_per_step = t['batch_size'] * t['grad_accum_steps'] * T
tokens_per_run = tokens_per_step * t['max_steps']
print(f'Approximate parameters: {params/1e6:.2f}M')
print(f'Tokens per optimizer step: {tokens_per_step:,}')
print(f'Tokens processed per run: {tokens_per_run/1e6:.2f}M')
print(f'Tokens / parameter per run: {tokens_per_run/params:.2f}')
print('Note: this is a compute-constrained scaling run, not a Chinchilla-optimal pretraining run.')